In [ ]:
#CNN(original version)
import os
import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import random

ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT, "data")
MODEL_DIR = os.path.join(ROOT, "models")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'[^A-Za-z0-9\u4e00-\u9fff ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def build_text_column(df: pd.DataFrame, duplicate_title=True, max_len=3000):
    title = df["title"].fillna("").map(normalize_text)
    content = df["content"].fillna("").map(normalize_text)
    if duplicate_title:
        text = title + " [SEP] " + title + " " + content
    else:
        text = title + " [SEP] " + content
    return text.str.slice(0, max_len)

def main():
    ds = load_dataset("sogou_news")
    train_df = ds["train"].to_pandas()
    test_df = ds["test"].to_pandas()

    train_df.to_csv(os.path.join(DATA_DIR, "sogou_train.csv"), index=False)
    test_df.to_csv(os.path.join(DATA_DIR, "sogou_test.csv"), index=False)

    train_texts = build_text_column(train_df)
    test_texts = build_text_column(test_df)

    y_train_full = train_df["label"].astype(int).values
    y_test = test_df["label"].astype(int).values

    X_train, X_val, y_train, y_val = train_test_split(
        train_texts, y_train_full, test_size=0.1, stratify=y_train_full, random_state=42
    )

    tokenizer = Tokenizer(num_words=50000, oov_token="[UNK]")
    tokenizer.fit_on_texts(X_train)

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_val_seq = tokenizer.texts_to_sequences(X_val)
    X_test_seq = tokenizer.texts_to_sequences(test_texts)

    max_len = 1000
    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

    vocab_size = min(50000, len(tokenizer.word_index) + 1)

    model = Sequential([
        Embedding(vocab_size, 256, input_length=max_len),
        Conv1D(128, 5, activation="relu"),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(64, activation="relu"),
        Dropout(0.5),
        Dense(len(np.unique(y_train_full)), activation="softmax")
    ])

    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    model.summary()

    early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

    model.fit(
        X_train_pad, y_train,
        epochs=10,
        batch_size=64,
        validation_data=(X_val_pad, y_val),
        callbacks=[early_stop],
        verbose=1
    )

    y_val_pred = np.argmax(model.predict(X_val_pad), axis=1)
    print("\nValidation Report")
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    y_test_pred = np.argmax(model.predict(X_test_pad), axis=1)
    print("\nTest Report")
    print(classification_report(y_test, y_test_pred))

    model_path = os.path.join(MODEL_DIR, "cnn_text_classifier.h5")
    model.save(model_path)
    print("\nModel saved to:", model_path)

    return test_df, y_test, y_test_pred

if __name__ == "__main__":
    test_df, y_test, y_test_pred = main()

label_map = {
    0: "Sports",
    1: "Finance",
    2: "Entertainment",
    3: "Automobile",
    4: "Technology"
}

samples = random.sample(range(len(test_df)), 5)
for i in samples:
    print("\n")
    print("Title:", test_df.iloc[i]['title'][:50])
    print("Content:", test_df.iloc[i]['content'][:120])
    print("True Label:", label_map.get(int(y_test[i]), str(y_test[i])))
    print("Predicted Label:", label_map.get(int(y_test_pred[i]), str(y_test_pred[i])))

In [ ]:
#TextCNN + BatchNormalization + update parameters
import os
import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding,Conv1D,GlobalMaxPooling1D,Dense,Dropout,Concatenate,BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
import random

ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT,"data")
MODEL_DIR = os.path.join(ROOT,"models")
os.makedirs(DATA_DIR,exist_ok=True)
os.makedirs(MODEL_DIR,exist_ok=True)

def normalize_text(s:str) ->str:
    if not isinstance(s,str):
        return ""
    s = re.sub(r'https?://\S+|www\.\S+',' ',s)
    s = re.sub(r'[^A-Za-z0-9\u4e00-\u9fff ]+',' ',s)
    s = re.sub(r'\s+',' ',s).strip()
    return s

def build_text_column(df: pd.DataFrame,duplicate_title=True,max_len=3000):
    title = df["title"].fillna("").map(normalize_text)
    content = df["content"].fillna("").map(normalize_text)
    if duplicate_title:
        text = title + " [SEP] " + title + " " + content
    else:
        text = title + " [SEP] " + content
    return text.str.slice(0,max_len)

def main():
    ds = load_dataset("sogou_news")
    train_df = ds["train"].to_pandas()
    test_df = ds["test"].to_pandas()

    train_df.to_csv(os.path.join(DATA_DIR, "sogou_train.csv"), index=False)
    test_df.to_csv(os.path.join(DATA_DIR, "sogou_test.csv"), index=False)

    train_texts = build_text_column(train_df)
    test_texts = build_text_column(test_df)

    y_train_full = train_df["label"].astype(int).values
    y_test = test_df["label"].astype(int).values

    X_train, X_val, y_train, y_val = train_test_split(
        train_texts, y_train_full, test_size=0.1, stratify=y_train_full, random_state=42
    )

    tokenizer = Tokenizer(num_words=50000, oov_token="[UNK]")
    tokenizer.fit_on_texts(X_train)

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_val_seq = tokenizer.texts_to_sequences(X_val)
    X_test_seq = tokenizer.texts_to_sequences(test_texts)

    max_len = 1000
    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

    vocab_size = min(50000, len(tokenizer.word_index) + 1)
    embedding_dim = 128
    num_classes = len(np.unique(y_train_full))
    filter_sizes = [3, 4, 5]
    num_filters = 256

    inputs = Input(shape=(max_len,))
    embedding = Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)

    conv_outputs = []
    for fs in filter_sizes:
        conv = Conv1D(filters=num_filters, kernel_size=fs, activation='relu')(embedding)
        conv = BatchNormalization()(conv)
        pool = GlobalMaxPooling1D()(conv)
        conv_outputs.append(pool)

    merged = Concatenate()(conv_outputs)
    drop = Dropout(0.3)(merged)
    dense = Dense(128, activation="relu")(drop)
    drop2 = Dropout(0.3)(dense)

    outputs = Dense(num_classes, activation="softmax")(drop2)

    model = Model(inputs, outputs)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    model.summary()

    early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

    model.fit(
        X_train_pad, y_train,
        epochs=10,
        batch_size=64,
        validation_data=(X_val_pad, y_val),
        callbacks=[early_stop],
        verbose=1
    )

    y_val_pred = np.argmax(model.predict(X_val_pad), axis=1)
    print("\nValidation Report")
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    y_test_pred = np.argmax(model.predict(X_test_pad), axis=1)
    print("\nTest Report")
    print(classification_report(y_test, y_test_pred))

    model_path = os.path.join(MODEL_DIR, "textcnn_optimized.h5")
    model.save(model_path)
    print("\nModel saved to:", model_path)

    return test_df, y_test, y_test_pred

if __name__ == "__main__":
    test_df, y_test, y_test_pred = main()

label_map = {
    0: "Sports",
    1: "Finance",
    2: "Entertainment",
    3: "Automobile",
    4: "Technology"
}

samples = random.sample(range(len(test_df)), 5)
for i in samples:
    print("\n")
    print("Title:", test_df.iloc[i]['title'][:50])
    print("Content:", test_df.iloc[i]['content'][:120])
    print("True Label:", label_map.get(int(y_test[i]), str(y_test[i])))
    print("Predicted Label:", label_map.get(int(y_test_pred[i]), str(y_test_pred[i])))